# Undertone AI Colab Runner

Run the setup cells from top to bottom. This notebook assumes the repository is stored in Google Drive at `/content/drive/MyDrive/undertone-ai-project/undertone-ai`, which matches the paths used by the app scripts.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_DIR = Path('/content/drive/MyDrive/undertone-ai-project/undertone-ai')
SCRIPTS_DIR = PROJECT_DIR / 'scripts'
DATA_DIR = PROJECT_DIR / 'data'

required_paths = [
    SCRIPTS_DIR / 'gradio_ver.py',
    SCRIPTS_DIR / 'makeup_recommender.py',
    DATA_DIR / 'makeup' / 'makeup_json',
    DATA_DIR / 'undertone',
    DATA_DIR / 'undertone_faces',
]

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Repo files were not found in the expected Google Drive location. '
        'Move/clone this repo to /content/drive/MyDrive/undertone-ai-project/undertone-ai. '
        f'Missing: {missing}'
    )

os.chdir(PROJECT_DIR)
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

print(f'Project directory: {PROJECT_DIR}')
print(f'Scripts directory: {SCRIPTS_DIR}')
print(f'Data directory: {DATA_DIR}')


In [ ]:
%pip install -q mediapipe==0.10.14 pillow-heif gradio opencv-python numpy torch torchvision torchaudio


## Run The App

This starts the Gradio webcam/upload app. Use the public `share=True` link printed by Gradio after the cell starts.


In [ ]:
import os

os.chdir(SCRIPTS_DIR)
!python gradio_ver.py


## Optional: Check Makeup Data Paths

Run this only if product images are not showing in the Gradio gallery.


In [ ]:
import os
from makeup_recommender import MAKEUP_DATABASE

base_makeup_path = DATA_DIR / 'makeup'
test_image = base_makeup_path / 'HUR_BLUSH' / '0000_ruby_red.webp'

print(f'Checking makeup path: {base_makeup_path}')
print(f'Makeup path exists: {base_makeup_path.exists()}')
print(f'Folders: {os.listdir(base_makeup_path) if base_makeup_path.exists() else []}')
print(f'Test image exists: {test_image.exists()}')
print(f'Items in database: {len(MAKEUP_DATABASE)}')


## Optional: Rebuild Undertone Face Patches

Run this only when you need to regenerate `data/undertone_faces` from `data/undertone`.


In [ ]:
import cv2
import os
import numpy as np
import mediapipe as mp

RAW_DATA = DATA_DIR / 'undertone'
OUT_DATA = DATA_DIR / 'undertone_faces'
IMG_SIZE = 224

mp_face = mp.solutions.face_detection

def crop_cheek_patch(img, bbox):
    h, w, _ = img.shape
    x, y, bw, bh = bbox

    x = int(x * w)
    y = int(y * h)
    bw = int(bw * w)
    bh = int(bh * h)

    cx = x + int(0.32 * bw)
    cy = y + int(0.42 * bh)

    half = IMG_SIZE // 2
    x1, y1 = max(cx - half, 0), max(cy - half, 0)
    x2, y2 = min(cx + half, w), min(cy + half, h)

    patch = img[y1:y2, x1:x2]

    if patch.shape[0] < 100 or patch.shape[1] < 100:
        return None

    return cv2.resize(patch, (IMG_SIZE, IMG_SIZE))

def extract_skin_patch(img):
    ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)

    lower = np.array([0, 133, 77], dtype=np.uint8)
    upper = np.array([255, 173, 127], dtype=np.uint8)

    mask = cv2.inRange(ycrcb, lower, upper)
    skin_ratio = np.sum(mask > 0) / (mask.shape[0] * mask.shape[1])
    if skin_ratio > 0.45:
        return cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    c = max(contours, key=cv2.contourArea)
    if cv2.contourArea(c) < 800:
        return None

    x, y, w, h = cv2.boundingRect(c)
    cx, cy = x + w // 2, y + h // 2
    half = IMG_SIZE // 2

    x1, y1 = max(cx - half, 0), max(cy - half, 0)
    x2, y2 = min(cx + half, img.shape[1]), min(cy + half, img.shape[0])

    patch = img[y1:y2, x1:x2]
    if patch.shape[0] < 100 or patch.shape[1] < 100:
        return None

    return cv2.resize(patch, (IMG_SIZE, IMG_SIZE))

kept, skipped = 0, 0

with mp_face.FaceDetection(model_selection=1, min_detection_confidence=0.6) as detector:
    for split in ['train', 'validation', 'test']:
        split_dir = RAW_DATA / split
        if not split_dir.exists():
            print(f'Skipping {split}: {split_dir} does not exist')
            continue

        for cls in os.listdir(split_dir):
            in_dir = split_dir / cls
            out_dir = OUT_DATA / split / cls
            out_dir.mkdir(parents=True, exist_ok=True)

            for img_name in os.listdir(in_dir):
                img_path = in_dir / img_name
                img = cv2.imread(str(img_path))

                if img is None:
                    skipped += 1
                    continue

                rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                results = detector.process(rgb)
                patch = None

                if results.detections:
                    bbox = results.detections[0].location_data.relative_bounding_box
                    patch = crop_cheek_patch(img, (bbox.xmin, bbox.ymin, bbox.width, bbox.height))

                if patch is None:
                    patch = extract_skin_patch(img)

                if patch is not None:
                    cv2.imwrite(str(out_dir / img_name), patch)
                    kept += 1
                else:
                    skipped += 1

print('SKIN PATCH PREPROCESSING COMPLETE')
print(f'Kept images:    {kept}')
print(f'Skipped images: {skipped}')
print(f'Output saved to:\n{OUT_DATA}')


## Optional: Train Undertone Model

Run this only if you need to retrain `data/undertone_faces/undertone_resnet18_best.pth`.


In [ ]:
import copy
import time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torchvision import datasets, models, transforms

base_path = DATA_DIR / 'undertone_faces'
data_dir = base_path

num_classes = 3
batch_size = 16
num_epochs = 30
learning_rate = 0.001

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0), ratio=(0.95, 1.05)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.20, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    'validation': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    'test': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
}

image_datasets = {}
dataloaders = {}

def has_images(path):
    for cls in path.iterdir():
        if cls.is_dir() and any(cls.iterdir()):
            return True
    return False

for split in ['train', 'validation', 'test']:
    path = data_dir / split

    if not path.exists():
        print(f'Skipping {split}: folder does not exist')
        continue

    if not has_images(path):
        print(f'Skipping {split}: no images found')
        continue

    image_datasets[split] = datasets.ImageFolder(str(path), transform=data_transforms[split])
    dataloaders[split] = torch.utils.data.DataLoader(
        image_datasets[split],
        batch_size=batch_size,
        shuffle=(split == 'train'),
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )

    print(f'{split}: {len(image_datasets[split])} images')

if 'train' not in image_datasets:
    raise RuntimeError('Training set is empty - cannot train.')
if 'validation' not in image_datasets:
    raise RuntimeError('Validation set is empty - cannot choose the best model.')

class_names = image_datasets['train'].classes
print('Classes:', class_names)

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=learning_rate)
scheduler = lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

def train_model(model, criterion, optimizer, scheduler, num_epochs):
    since = time.time()
    best_acc = 0.0
    best_wts = copy.deepcopy(model.state_dict())

    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch}/{num_epochs - 1}')
        print('-' * 20)

        for phase in ['train', 'validation']:
            model.train() if phase == 'train' else model.eval()
            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / len(image_datasets[phase])
            epoch_acc = running_corrects.double() / len(image_datasets[phase])

            print(f'{phase:12} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'validation' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_wts = copy.deepcopy(model.state_dict())

    time_elapsed = time.time() - since
    print(f'\nTraining finished in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best validation accuracy: {best_acc:.4f}')

    model.load_state_dict(best_wts)
    return model

model = train_model(model, criterion, optimizer, scheduler, num_epochs)

save_path = data_dir / 'undertone_resnet18_best.pth'
torch.save(model.state_dict(), save_path)
print(f'Model saved to:\n{save_path}')
